In [1]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OrdinalEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [2]:
FEATURE_PATH = "data/processed/CA_1_features_v1.parquet"

MODEL_DIR = Path("models")
RESULT_DIR = Path("results")

MODEL_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)

con = duckdb.connect()

In [3]:
df = con.execute(f"""
SELECT
    item_id,
    dept_id,
    cat_id,
    store_id,
    state_id,
    date,
    units_sold,
    sell_price,
    weekday,
    month,
    year,
    event_name_1,
    event_type_1,
    snap_CA,
    lag_1,
    lag_7,
    lag_28,
    rolling_mean_7,
    rolling_mean_28
FROM read_parquet('{FEATURE_PATH}')
WHERE
    is_available = 1
    AND lag_28 IS NOT NULL
""").fetchdf()

print("Shape:", df.shape)
display(df.head())

Shape: (4751349, 19)


,item_id,dept_id,cat_id,store_id,state_id,date,units_sold,sell_price,weekday,month,year,event_name_1,event_type_1,snap_CA,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,2013-02-16,0,6.98,Saturday,2,2013,None,None,0,3,4,1,2.428571,1.392857
1,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,2013-02-17,0,6.98,Sunday,2,2013,None,None,0,0,3,0,1.857143,1.357143
2,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,2013-02-18,0,6.98,Monday,2,2013,PresidentsDay,National,0,0,1,1,1.428571,1.357143
3,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,2013-02-19,0,6.98,Tuesday,2,2013,None,None,0,0,2,1,1.285714,1.321429
4,FOODS_2_083,FOODS_2,FOODS,CA_1,CA,2013-02-20,1,6.98,Wednesday,2,2013,LentWeek2,Religious,0,0,1,0,1.000000,1.285714


In [4]:
max_date = df["date"].max()

test_start = max_date - pd.Timedelta(days=27)
val_start = test_start - pd.Timedelta(days=28)

train = df[
    df["date"] < val_start
].copy()

val = df[
    (df["date"] >= val_start) &
    (df["date"] < test_start)
].copy()

test = df[
    df["date"] >= test_start
].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print("\nValidation start:", val_start)
print("Test start:", test_start)
print("Final date:", max_date)

Train: (4580605, 19)
Validation: (85372, 19)
Test: (85372, 19)

Validation start: 2016-03-28 00:00:00
Test start: 2016-04-25 00:00:00
Final date: 2016-05-22 00:00:00


In [5]:
target = "units_sold"

features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sell_price",
    "weekday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

cat_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1"
]

print("Number of features:", len(features))

Number of features: 17


In [6]:
# Lag 1
mae_1 = mean_absolute_error(
    val[target],
    val["lag_1"]
)

rmse_1 = np.sqrt(
    mean_squared_error(
        val[target],
        val["lag_1"]
    )
)


# Lag 7
mae_7 = mean_absolute_error(
    val[target],
    val["lag_7"]
)

rmse_7 = np.sqrt(
    mean_squared_error(
        val[target],
        val["lag_7"]
    )
)


# Rolling Mean 28
mae_roll = mean_absolute_error(
    val[target],
    val["rolling_mean_28"]
)

rmse_roll = np.sqrt(
    mean_squared_error(
        val[target],
        val["rolling_mean_28"]
    )
)


print("Lag 1")
print("MAE:", mae_1)
print("RMSE:", rmse_1)

print("\nLag 7")
print("MAE:", mae_7)
print("RMSE:", rmse_7)

print("\nRolling Mean 28")
print("MAE:", mae_roll)
print("RMSE:", rmse_roll)

Lag 1
MAE: 1.2817317153164973
RMSE: 2.768170875297637

Lag 7
MAE: 1.3096799887550954
RMSE: 2.8065592077338195

Rolling Mean 28
MAE: 1.087551706481215
RMSE: 2.193423582379451


In [7]:
CATBOOST_PATH = MODEL_DIR / "catboost_ca1.cbm"

if not CATBOOST_PATH.exists():
    raise FileNotFoundError(
        f"CatBoost model not found at {CATBOOST_PATH}. "
        "Save it from notebook 05 first."
    )

cat_model = CatBoostRegressor()
cat_model.load_model(str(CATBOOST_PATH))

print("CatBoost loaded:", cat_model.is_fitted())

CatBoost loaded: True


In [8]:
X_val_cat = val[features].copy()

for col in cat_cols:
    X_val_cat[col] = (
        X_val_cat[col]
        .fillna("None")
        .astype(str)
    )

cat_pred = cat_model.predict(X_val_cat)

cat_mae = mean_absolute_error(
    val[target],
    cat_pred
)

cat_rmse = np.sqrt(
    mean_squared_error(
        val[target],
        cat_pred
    )
)

print("CatBoost MAE:", cat_mae)
print("CatBoost RMSE:", cat_rmse)

CatBoost MAE: 1.0384569631641956
CatBoost RMSE: 2.0128241533877698


In [9]:
X_train = train[features].copy()
X_val = val[features].copy()

y_train = train[target]
y_val = val[target]

for col in cat_cols:
    X_train[col] = X_train[col].fillna("None").astype(str)
    X_val[col] = X_val[col].fillna("None").astype(str)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[cat_cols] = encoder.fit_transform(
    X_train[cat_cols]
)

X_val[cat_cols] = encoder.transform(
    X_val[cat_cols]
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (4580605, 17)
X_val: (85372, 17)


In [10]:
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-rmse:3.36235
[100]	validation_0-rmse:2.02809
[200]	validation_0-rmse:2.02734
[300]	validation_0-rmse:2.02473
[400]	validation_0-rmse:2.02289
[499]	validation_0-rmse:2.02256


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=-1, num_parallel_tree=None, ...)

In [11]:
xgb_pred = xgb_model.predict(X_val)

xgb_mae = mean_absolute_error(
    y_val,
    xgb_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        xgb_pred
    )
)

print("XGBoost MAE:", xgb_mae)
print("XGBoost RMSE:", xgb_rmse)

XGBoost MAE: 1.0395276546478271
XGBoost RMSE: 2.0225616091102108


In [12]:
xgb_model.save_model(
    str(MODEL_DIR / "xgboost_ca1.json")
)

print("XGBoost saved.")

XGBoost saved.


In [13]:
lgb_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)]
)

/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060547 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1363
[LightGBM] [Info] Number of data points in the train set: 4580605, number of used features: 15
[LightGBM] [Info] Start training from score 1.632761


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=500,
              n_jobs=-1, num_leaves=63, random_state=42, subsample=0.8)

In [14]:
lgb_pred = lgb_model.predict(X_val)

lgb_mae = mean_absolute_error(
    y_val,
    lgb_pred
)

lgb_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        lgb_pred
    )
)

print("LightGBM MAE:", lgb_mae)
print("LightGBM RMSE:", lgb_rmse)

LightGBM MAE: 1.0412838747419002
LightGBM RMSE: 2.0212272592467393


In [15]:
lgb_model.booster_.save_model(
    str(MODEL_DIR / "lightgbm_ca1.txt")
)

print("LightGBM saved.")

LightGBM saved.


In [16]:
comparison = pd.DataFrame({
    "model": [
        "Lag 1",
        "Lag 7",
        "Rolling Mean 28",
        "CatBoost",
        "XGBoost",
        "LightGBM"
    ],

    "MAE": [
        mae_1,
        mae_7,
        mae_roll,
        cat_mae,
        xgb_mae,
        lgb_mae
    ],

    "RMSE": [
        rmse_1,
        rmse_7,
        rmse_roll,
        cat_rmse,
        xgb_rmse,
        lgb_rmse
    ]
})

comparison = comparison.sort_values(
    "RMSE"
).reset_index(drop=True)

display(comparison)

,model,MAE,RMSE
0,CatBoost,1.038457,2.012824
1,LightGBM,1.041284,2.021227
2,XGBoost,1.039528,2.022562
3,Rolling Mean 28,1.087552,2.193424
4,Lag 1,1.281732,2.768171
5,Lag 7,1.309680,2.806559


In [17]:
best = comparison.iloc[0]

print("BEST MODEL")
print("----------------")
print("Model:", best["model"])
print("MAE:", round(best["MAE"], 4))
print("RMSE:", round(best["RMSE"], 4))

BEST MODEL
----------------
Model: CatBoost
MAE: 1.0385
RMSE: 2.0128


In [18]:
best_baseline_rmse = rmse_roll

improvement_pct = (
    (best_baseline_rmse - best["RMSE"])
    / best_baseline_rmse
) * 100

print(
    "RMSE improvement over Rolling Mean 28:",
    round(improvement_pct, 2),
    "%"
)

RMSE improvement over Rolling Mean 28: 8.23 %


In [19]:
comparison.to_csv(
    RESULT_DIR / "ca1_model_comparison.csv",
    index=False
)

print("Results saved.")

Results saved.


In [20]:
xgb_importance = pd.DataFrame({
    "feature": features,
    "importance": xgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(xgb_importance.head(15))

,feature,importance
15,rolling_mean_7,0.697315
16,rolling_mean_28,0.142581
12,lag_1,0.034035
13,lag_7,0.022120
6,weekday,0.017398
2,cat_id,0.016060
14,lag_28,0.015493
9,event_name_1,0.008642
5,sell_price,0.008083
11,snap_CA,0.008066


In [21]:
lgb_importance = pd.DataFrame({
    "feature": features,
    "importance": lgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(lgb_importance.head(15))

,feature,importance
12,lag_1,4696
15,rolling_mean_7,4397
14,lag_28,4228
13,lag_7,3282
16,rolling_mean_28,2847
6,weekday,2457
5,sell_price,2319
0,item_id,2182
7,month,1817
8,year,1095


In [22]:
cat_importance = pd.DataFrame({
    "feature": features,
    "importance": cat_model.get_feature_importance()
}).sort_values(
    "importance",
    ascending=False
)

display(cat_importance.head(15))

,feature,importance
15,rolling_mean_7,37.249885
16,rolling_mean_28,19.432287
12,lag_1,16.713716
6,weekday,9.268159
13,lag_7,4.501403
14,lag_28,4.134492
5,sell_price,2.763078
0,item_id,1.381718
9,event_name_1,1.164371
7,month,1.076189
